# Sanity checks

Two checks that have to pass before any outcome is trusted, both with thresholds fixed in `docs/experiment_design.md` before this notebook was run.

1. **Sample ratio mismatch (SRM).** Did randomization and logging produce the one third split the experiment was designed for? Fails at p < 0.0005 (Fabijan et al. 2019).
2. **Covariate balance.** Do the arms look alike on pre-treatment characteristics? Any absolute standardized mean difference of 0.1 or more is flagged (Austin 2009).

Neither check touches visit, conversion, or spend.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
from scipy import stats
from load import load_hillstrom
from checks import srm_test, smd, balance_table

df = load_hillstrom()

SRM_THRESHOLD = 0.0005
SMD_THRESHOLD = 0.1
TABLES = Path.cwd().parent / "results" / "tables"
TABLES.mkdir(parents=True, exist_ok=True)

print(f"rows: {len(df)}")
print(f"SRM threshold: p < {SRM_THRESHOLD}, SMD threshold: |SMD| >= {SMD_THRESHOLD}")

rows: 64000
SRM threshold: p < 0.0005, SMD threshold: |SMD| >= 0.1


## 1. SRM chi-square test

The design split customers one third each, so the expected count per arm is total / 3. The chi-square statistic adds up how far each observed count sits from its expected count, scaled by the expected count:

chi2 = sum over arms of (observed - expected)^2 / expected

With 3 arms there are 3 - 1 = 2 degrees of freedom. The p-value is the chance of a chi2 this large or larger if the split really were one third each.

Computing each piece by hand first.

In [2]:
observed = df["segment"].value_counts().sort_index()
n_total = observed.sum()
expected = pd.Series(n_total / 3, index=observed.index)
contribution = (observed - expected) ** 2 / expected

srm_by_hand = pd.DataFrame({
    "observed": observed,
    "expected": expected,
    "observed_minus_expected": observed - expected,
    "observed_share": observed / n_total,
    "chi2_contribution": contribution,
})
print(f"total customers: {n_total:,}")
srm_by_hand

total customers: 64,000


,observed,expected,observed_minus_expected,observed_share,chi2_contribution
segment,,,,,
Mens E-Mail,21307,21333.333333,-26.333333,0.332922,0.032505
No E-Mail,21306,21333.333333,-27.333333,0.332906,0.035021
Womens E-Mail,21387,21333.333333,53.666667,0.334172,0.135005


In [3]:
chi2_by_hand = contribution.sum()
dof = len(observed) - 1
p_by_hand = stats.chi2.sf(chi2_by_hand, dof)

print(f"chi2 statistic: {chi2_by_hand:.6f}")
print(f"degrees of freedom: {dof}")
print(f"p-value: {p_by_hand:.6f}")

chi2 statistic: 0.202531
degrees of freedom: 2
p-value: 0.903693


Cross-checking the hand calculation against the `srm_test` helper in `src/checks.py` and against scipy's built-in `chisquare`. All three should agree.

In [4]:
helper = srm_test(observed)
scipy_check = stats.chisquare(observed, expected)

print(f"helper:  chi2 {helper['chi2']:.6f}, p {helper['p_value']:.6f}")
print(f"scipy:   chi2 {scipy_check.statistic:.6f}, p {scipy_check.pvalue:.6f}")
print()
srm_pass = p_by_hand >= SRM_THRESHOLD
print(f"SRM verdict: {'PASS, no sample ratio mismatch' if srm_pass else 'FAIL, stop and investigate'} (p {p_by_hand:.6f} vs threshold {SRM_THRESHOLD})")

helper:  chi2 0.202531, p 0.903693
scipy:   chi2 0.202531, p 0.903693

SRM verdict: PASS, no sample ratio mismatch (p 0.903693 vs threshold 0.0005)


In [5]:
srm_out = srm_by_hand.copy()
srm_out["chi2_total"] = chi2_by_hand
srm_out["dof"] = dof
srm_out["p_value"] = p_by_hand
srm_out["threshold"] = SRM_THRESHOLD
srm_out["pass"] = srm_pass
srm_out.to_csv(TABLES / "srm_check.csv", index_label="arm")
print("saved results/tables/srm_check.csv")

saved results/tables/srm_check.csv


## 2. Covariate balance

### What chance alone produces

Even with perfect randomization the arms won't match exactly. For a covariate with standard deviation sigma, the difference in arm means has variance sigma^2/n_a + sigma^2/n_b. Dividing by sigma to put it on the SMD scale, the typical SMD from chance alone is about

SE(SMD) = sqrt(1/n_a + 1/n_b)

This depends only on arm sizes. Comparing it to the 0.1 threshold shows how far outside normal noise a flagged covariate would be.

In [6]:
n = df["segment"].value_counts()
n_min_pair = n.nsmallest(2)
se_smd_chance = np.sqrt(1 / n_min_pair.iloc[0] + 1 / n_min_pair.iloc[1])

print(f"two smallest arms: {n_min_pair.to_dict()}")
print(f"chance-level SE of SMD: {se_smd_chance:.6f}")
print(f"0.1 threshold in chance-level standard errors: {SMD_THRESHOLD / se_smd_chance:.2f}")

two smallest arms: {'No E-Mail': 21306, 'Mens E-Mail': 21307}
chance-level SE of SMD: 0.009689
0.1 threshold in chance-level standard errors: 10.32


### history skew check

SMD is built on means and variances, so a few very large values can move it. Checking the pooled shape of `history` (all arms together, a pre-treatment column), and whether a log transform is possible (needs every value above zero).

In [7]:
print(df["history"].describe())
print()
print(f"skewness: {stats.skew(df['history']):.4f}")
print(f"skewness of log(history): {stats.skew(np.log(df['history'])):.4f}" if (df["history"] > 0).all() else "history has zeros, log not defined")
print(f"min history: {df['history'].min()}, any values <= 0: {(df['history'] <= 0).any()}")

count    64000.000000
mean       242.085656
std        256.158608
min         29.990000
25%         64.660000
50%        158.110000
75%        325.657500
max       3345.930000
Name: history, dtype: float64

skewness: 2.4237
skewness of log(history): -0.0114
min history: 29.99, any values <= 0: False


### Building the covariate set

Numeric covariates go in as they are. zip_code and channel are categories, so each level becomes its own 0/1 indicator, and its SMD compares the share of customers in that level between arms. `log_history` is included as the robustness view on `history`, not a gate.

In [8]:
cov = df[["segment", "recency", "history", "mens", "womens", "newbie"]].copy()
cov["log_history"] = np.log(df["history"])
for col in ["zip_code", "channel"]:
    dummies = pd.get_dummies(df[col], prefix=col).astype(int)
    cov = pd.concat([cov, dummies], axis=1)

covariates = [c for c in cov.columns if c != "segment"]
print(f"{len(covariates)} covariates: {covariates}")

12 covariates: ['recency', 'history', 'mens', 'womens', 'newbie', 'log_history', 'zip_code_Rural', 'zip_code_Surburban', 'zip_code_Urban', 'channel_Multichannel', 'channel_Phone', 'channel_Web']


### SMD formula, worked once by hand

For two arms a and b,

SMD = (mean_a - mean_b) / sqrt( (var_a + var_b) / 2 )

The denominator is the pooled standard deviation of the two arms (Austin 2009's form). Working it through for `history`, Mens E-Mail vs No E-Mail, showing every piece.

In [9]:
a = cov.loc[cov["segment"] == "Mens E-Mail", "history"]
b = cov.loc[cov["segment"] == "No E-Mail", "history"]

mean_a, mean_b = a.mean(), b.mean()
var_a, var_b = a.var(ddof=1), b.var(ddof=1)
pooled_sd = np.sqrt((var_a + var_b) / 2)
smd_by_hand = (mean_a - mean_b) / pooled_sd

print(f"n_a {len(a):,}, n_b {len(b):,}")
print(f"mean_a {mean_a:.4f}, mean_b {mean_b:.4f}, difference {mean_a - mean_b:.4f}")
print(f"var_a {var_a:.4f}, var_b {var_b:.4f}")
print(f"pooled SD {pooled_sd:.4f}")
print(f"SMD by hand {smd_by_hand:.6f}")
print(f"SMD from helper {smd(a, b):.6f}")

n_a 21,307, n_b 21,306
mean_a 242.8359, mean_b 240.8827, difference 1.9533
var_a 67785.0827, var_b 63877.1850
pooled SD 256.5758
SMD by hand 0.007613
SMD from helper 0.007613


### Full balance table

Every covariate, every pair of arms, using the same formula through the `balance_table` helper.

In [10]:
balance = balance_table(cov, "segment", covariates, threshold=SMD_THRESHOLD)
balance["abs_smd"] = balance["smd"].abs()
balance["smd_in_chance_se"] = balance["abs_smd"] / se_smd_chance

pd.set_option("display.max_rows", 100)
balance.style.format({"mean_a": "{:.4f}", "mean_b": "{:.4f}", "smd": "{:.4f}", "abs_smd": "{:.4f}", "smd_in_chance_se": "{:.2f}"})

,pair,covariate,mean_a,mean_b,smd,flagged,abs_smd,smd_in_chance_se
0,Mens E-Mail vs No E-Mail,recency,5.7736,5.7497,0.0068,False,0.0068,0.71
1,Mens E-Mail vs No E-Mail,history,242.8359,240.8827,0.0076,False,0.0076,0.79
2,Mens E-Mail vs No E-Mail,mens,0.5509,0.5532,-0.0046,False,0.0046,0.47
3,Mens E-Mail vs No E-Mail,womens,0.5514,0.5476,0.0076,False,0.0076,0.78
4,Mens E-Mail vs No E-Mail,newbie,0.5015,0.5020,-0.0009,False,0.0009,0.09
5,Mens E-Mail vs No E-Mail,log_history,4.9927,4.9939,-0.0012,False,0.0012,0.12
6,Mens E-Mail vs No E-Mail,zip_code_Rural,0.1522,0.1473,0.0137,False,0.0137,1.41
7,Mens E-Mail vs No E-Mail,zip_code_Surburban,0.4459,0.4518,-0.0117,False,0.0117,1.21
8,Mens E-Mail vs No E-Mail,zip_code_Urban,0.4019,0.4009,0.0020,False,0.0020,0.20
9,Mens E-Mail vs No E-Mail,channel_Multichannel,0.1209,0.1223,-0.0042,False,0.0042,0.43


In [11]:
print(f"largest |SMD|: {balance['abs_smd'].max():.4f} ({balance.loc[balance['abs_smd'].idxmax(), 'covariate']}, {balance.loc[balance['abs_smd'].idxmax(), 'pair']})")
print(f"covariates flagged (|SMD| >= {SMD_THRESHOLD}): {int(balance['flagged'].sum())} of {len(balance)} checks")
print()
print("history vs log_history, robustness view:")
print(balance[balance["covariate"].isin(["history", "log_history"])][["pair", "covariate", "smd"]].to_string(index=False))

balance.to_csv(TABLES / "covariate_balance.csv", index=False)
print()
print("saved results/tables/covariate_balance.csv")

largest |SMD|: 0.0169 (channel_Phone, Mens E-Mail vs Womens E-Mail)
covariates flagged (|SMD| >= 0.1): 0 of 36 checks

history vs log_history, robustness view:
                        pair   covariate       smd
    Mens E-Mail vs No E-Mail     history  0.007613
    Mens E-Mail vs No E-Mail log_history -0.001191
Mens E-Mail vs Womens E-Mail     history  0.001161
Mens E-Mail vs Womens E-Mail log_history -0.007647
  No E-Mail vs Womens E-Mail     history -0.006511
  No E-Mail vs Womens E-Mail log_history -0.006475

saved results/tables/covariate_balance.csv
